In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_features.csv")

rfm = df[
    ["customer_id", "recency_days", "frequency", "monetary_value"]
].copy()

rfm = rfm.dropna(
    subset=["recency_days", "frequency", "monetary_value"]
)

rfm = rfm[
    (rfm["frequency"] > 0) &
    (rfm["monetary_value"] > 0)
].copy()

rfm.head()

,customer_id,recency_days,frequency,monetary_value
0,CUST-000001,7.0,8,9469.55
1,CUST-000002,283.0,10,18479.69
2,CUST-000003,210.0,7,8268.66
3,CUST-000004,55.0,6,11092.52
4,CUST-000005,413.0,8,10063.97


In [2]:
rfm[["recency_days", "frequency", "monetary_value"]].describe(
    percentiles=[0.20, 0.40, 0.60, 0.80]
)

,recency_days,frequency,monetary_value
count,24876.000000,24876.000000,24876.000000
mean,322.816289,5.214584,7240.084967
std,320.099733,2.239154,4264.985663
min,1.000000,1.000000,13.930000
20%,54.000000,3.000000,3500.640000
40%,153.000000,4.000000,5548.250000
50%,223.000000,5.000000,6610.855000
60%,309.000000,6.000000,7716.290000
80%,539.000000,7.000000,10643.530000
max,1825.000000,17.000000,34083.400000


In [3]:
thresholds = rfm[
    ["recency_days", "frequency", "monetary_value"]
].quantile([0.20, 0.40, 0.60, 0.80])

thresholds

,recency_days,frequency,monetary_value
0.2,54.0,3.0,3500.64
0.4,153.0,4.0,5548.25
0.6,309.0,6.0,7716.29
0.8,539.0,7.0,10643.53


In [4]:
# Recency: fewer days since purchase is better
rfm["R_score"] = np.ceil(
    rfm["recency_days"].rank(
        ascending=False, method="average", pct=True
    ) * 5
).clip(1, 5).astype(int)

# Frequency: more orders is better
rfm["F_score"] = np.ceil(
    rfm["frequency"].rank(
        ascending=True, method="average", pct=True
    ) * 5
).clip(1, 5).astype(int)

# Monetary: higher spending is better
rfm["M_score"] = np.ceil(
    rfm["monetary_value"].rank(
        ascending=True, method="average", pct=True
    ) * 5
).clip(1, 5).astype(int)

rfm["RFM_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

rfm.head()

,customer_id,recency_days,frequency,monetary_value,R_score,F_score,M_score,RFM_score
0,CUST-000001,7.0,8,9469.55,5,5,4,554
1,CUST-000002,283.0,10,18479.69,3,5,5,355
2,CUST-000003,210.0,7,8268.66,3,4,4,344
3,CUST-000004,55.0,6,11092.52,4,4,5,445
4,CUST-000005,413.0,8,10063.97,2,5,4,254


In [5]:
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "High-value loyal customers"

    elif r >= 4 and f <= 2:
        return "Recent customers"

    elif r >= 3 and f >= 3 and m >= 3:
        return "Potential loyalists"

    elif r <= 2 and f >= 3:
        return "Customers at risk of inactivity"

    elif r <= 2 and f <= 2 and m <= 2:
        return "Inactive customers"

    else:
        return "Regular / needs attention"


rfm["customer_group"] = rfm.apply(assign_segment, axis=1)

rfm[
    ["customer_id", "R_score", "F_score", "M_score",
     "RFM_score", "customer_group"]
].head(10)

,customer_id,R_score,F_score,M_score,RFM_score,customer_group
0,CUST-000001,5,5,4,554,High-value loyal customers
1,CUST-000002,3,5,5,355,Potential loyalists
2,CUST-000003,3,4,4,344,Potential loyalists
3,CUST-000004,4,4,5,445,High-value loyal customers
4,CUST-000005,2,5,4,254,Customers at risk of inactivity
5,CUST-000006,3,4,5,345,Potential loyalists
6,CUST-000007,2,2,4,224,Regular / needs attention
7,CUST-000008,4,4,3,443,Potential loyalists
8,CUST-000009,2,1,1,211,Inactive customers
9,CUST-000010,4,2,4,424,Recent customers


In [6]:
segment_summary = rfm.groupby("customer_group").agg(
    customers=("customer_id", "nunique"),
    avg_recency_days=("recency_days", "mean"),
    median_recency_days=("recency_days", "median"),
    avg_frequency=("frequency", "mean"),
    avg_monetary_value=("monetary_value", "mean"),
    median_monetary_value=("monetary_value", "median")
).round(2)

segment_summary["customer_pct"] = (
    segment_summary["customers"] / rfm["customer_id"].nunique() * 100
).round(2)

segment_summary.sort_values("customers", ascending=False)

,customers,avg_recency_days,median_recency_days,avg_frequency,avg_monetary_value,median_monetary_value,customer_pct
customer_group,,,,,,,
Regular / needs attention,4850,312.00,236.0,4.19,5378.63,5099.31,19.50
Potential loyalists,4766,152.59,166.0,6.43,9124.03,8001.06,19.16
Customers at risk of inactivity,4269,515.27,455.0,6.17,8534.27,8074.40,17.16
Inactive customers,4261,742.60,672.0,2.70,2869.05,2870.10,17.13
High-value loyal customers,4067,61.06,52.0,7.86,12185.45,11441.81,16.35
Recent customers,2663,66.72,58.0,3.34,4625.16,4208.03,10.71


In [7]:
for col in ["R_score", "F_score", "M_score"]:
    print(f"\n{col} distribution:")
    print(rfm[col].value_counts().sort_index())


R_score distribution:
R_score
1    4972
2    4986
3    4985
4    4938
5    4995
Name: count, dtype: int64

F_score distribution:
F_score
1    5790
2    4273
3    4373
4    6655
5    3785
Name: count, dtype: int64

M_score distribution:
M_score
1    4975
2    4975
3    4975
4    4975
5    4976
Name: count, dtype: int64


In [8]:
rfm.to_csv("customer_rfm_segments.csv", index=False)

segment_summary.to_csv("rfm_segment_summary.csv")

print("RFM analysis completed.")

RFM analysis completed.
